In [4]:
import psycopg2
from psycopg2 import sql

conn = psycopg2.connect(dbname='wikidata_change_classification', user='postgres', password='postgres', host='172.16.64.23', port='5432')

queries = [
    ("Number of files", 
     "SELECT COUNT(distinct file_path) FROM revision"),

    ("Number of changes", 
     "SELECT COUNT(*) FROM value_change"),
    
    ("Number of creates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'CREATE'"),

    ("Number of deletes", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'DELETE'"),

    ("Number of updates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'UPDATE'"),
    
    ("Number of value updates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'UPDATE' AND change_target = ''"),
    
    ("Number of string updates", 
     """SELECT COUNT(*) FROM value_change 
        WHERE action = 'UPDATE' AND change_target = '' 
        AND datatype IN ('monolingualtext', 'string', 'external-id', 'url', 'commonsMedia', 
                         'geo-shape', 'tabular-data', 'math', 'musical-notation', 'unknown-values')"""),
    
    ("Number of entity updates", 
     """SELECT COUNT(*) FROM value_change 
        WHERE action = 'UPDATE' AND change_target = '' 
        AND datatype IN ('wikibase-item', 'wikibase-entityid', 'wikibase-property', 
                         'wikibase-lexeme', 'wikibase-sense', 'wikibase-form', 'entity-schema')"""),

    ("Number of quantity updates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'UPDATE' AND change_target = '' AND datatype = 'quantity'"),
    
    ("Number of time updates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'UPDATE' AND change_target = '' AND datatype = 'time'"),
    
    ("Number of globecoordinate updates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'UPDATE' AND change_target = '' AND datatype = 'globecoordinate'"),

    ("Number of rank updates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'UPDATE' AND change_target = 'rank'"),

    ("Number of rank creations", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'CREATE' AND change_target = 'rank'"),

    ("Number of rank deletions", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'DELETE' AND change_target = 'rank'"),
    
    ("Number of dt metadata updates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'UPDATE' AND change_target != 'rank' AND change_target != ''"),

    ("Number of dt metadata creations", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'CREATE' AND change_target != 'rank' AND change_target != ''"),

    ("Number of dt metadata deletions", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'DELETE' AND change_target != 'rank' AND change_target != ''"),
    
]

print("="*70)
print("DATABASE STATISTICS")
print("="*70)

try:
    cur = conn.cursor()
    
    for description, query in queries:
        cur.execute(query)
        result = cur.fetchone()[0]
        print(f"{description:.<50} {result:>15,}")
    
    cur.close()
    
except Exception as e:
    print(f"Error: {e}")
    
finally:
    conn.close()

print("="*70)


DATABASE STATISTICS
Number of files...................................             107
Number of changes.................................      59,641,183
Number of creates.................................      47,790,058
Number of deletes.................................       7,911,307
Number of updates.................................       3,939,818
Number of value updates...........................       2,760,666
Number of string updates..........................       1,550,206
Number of entity updates..........................         956,018
Number of quantity updates........................         121,727
Number of time updates............................         105,357
Number of globecoordinate updates.................          27,025
Number of rank updates............................         558,610
Number of rank creations..........................      21,151,877
Number of rank deletions..........................       3,457,076
Number of dt metadata updates.............

In [ ]:
df = pd.read_csv('../experiments/20251112_174658/cluster_examples_string.csv')
df['change_target'] = df['change_target'].astype(str).fillna('')
df['revision_id'] = df['revision_id'].astype(str)
df['property_id'] = df['property_id'].astype(int)
df['value_id'] = df['value_id'].astype(str)

In [33]:
df_features = pd.read_parquet('features/string_features.parquet')

df_features['change_target'] = df_features['change_target'].astype(str).fillna('')
df_features['revision_id'] = df_features['revision_id'].astype(str)
df_features['property_id'] = df_features['property_id'].astype(int)
df_features['value_id'] = df_features['value_id'].astype(str)


df_cluster_id = pd.merge(
    df_features,
    df[['revision_id', 'property_id', 'value_id', 'change_target', 'cluster_id']],
    on=['revision_id', 'property_id', 'value_id', 'change_target'],
    how='inner'
)

df_cluster_id.head()



,levenshtein_distance,length_diff_abs,case_differs,spaces_differs,punct_differs,hyph_dash_differs,brackets_differs,token_count_old,token_count_new,token_overlap,...,day_of_week_encoded,hour_of_day_encoded,is_weekend_encoded,num_changes_in_revision,entity_age_days,revision_id,property_id,value_id,change_target,cluster_id


In [34]:
# SHOW ME THIS OUTPUT
print("=" * 50)
print("DF_FEATURES:")
print(f"Shape: {df_features.shape}")
print(f"Columns: {df_features.columns.tolist()}")
print(df_features[['revision_id', 'property_id', 'value_id', 'change_target']].head(3))
print(df_features[['revision_id', 'property_id', 'value_id', 'change_target']].dtypes)

print("\n" + "=" * 50)
print("DF (cluster assignments):")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(df[['revision_id', 'property_id', 'value_id', 'change_target', 'cluster_id']].head(3))
print(df[['revision_id', 'property_id', 'value_id', 'change_target']].dtypes)

print("\n" + "=" * 50)
print("AFTER MERGE:")
print(f"Shape: {df_cluster_id.shape}")

print("\n" + "=" * 50)
print("DO ANY REVISION_IDS MATCH?")
common = set(df_features['revision_id'].head(100)) & set(df['revision_id'].head(100))
print(f"Common revision_ids in first 100: {len(common)}")
print(f"Sample common: {list(common)[:5]}")

DF_FEATURES:
Shape: (494396, 24)
Columns: ['levenshtein_distance', 'length_diff_abs', 'case_differs', 'spaces_differs', 'punct_differs', 'hyph_dash_differs', 'brackets_differs', 'token_count_old', 'token_count_new', 'token_overlap', 'old_in_new', 'new_in_old', 'edit_distance_ratio', 'semantic_similarity', 'user_type_encoded', 'day_of_week_encoded', 'hour_of_day_encoded', 'is_weekend_encoded', 'num_changes_in_revision', 'entity_age_days', 'revision_id', 'property_id', 'value_id', 'change_target']
  revision_id  property_id                                    value_id  \
0   212576787          214   Q974$EC98752F-DBC3-45BF-A394-835D357ABE33   
1   212576789          214   Q983$CADDEE1A-0E4D-441F-B67B-EEE2B4DFA345   
2   212576791          214  Q1007$672DBD99-F343-4599-8A6D-A3425CE20791   

  change_target  
0                
1                
2                
revision_id      object
property_id       int64
value_id         object
change_target    object
dtype: object

DF (cluster assignm

In [12]:
feature_cols = df_features.columns

# Show mean of each feature per cluster
for feature in feature_cols:
    print(f"\n=== {feature} ===")
    print(df_cluster_id.groupby('cluster_id')[feature].describe())


=== levenshtein_distance ===
Empty DataFrame
Columns: [count, mean, std, min, 25%, 50%, 75%, max]
Index: []

=== length_diff_abs ===
Empty DataFrame
Columns: [count, mean, std, min, 25%, 50%, 75%, max]
Index: []

=== case_differs ===
Empty DataFrame
Columns: [count, mean, std, min, 25%, 50%, 75%, max]
Index: []

=== spaces_differs ===
Empty DataFrame
Columns: [count, mean, std, min, 25%, 50%, 75%, max]
Index: []

=== punct_differs ===
Empty DataFrame
Columns: [count, mean, std, min, 25%, 50%, 75%, max]
Index: []

=== hyph_dash_differs ===
Empty DataFrame
Columns: [count, mean, std, min, 25%, 50%, 75%, max]
Index: []

=== brackets_differs ===
Empty DataFrame
Columns: [count, mean, std, min, 25%, 50%, 75%, max]
Index: []

=== token_count_old ===
Empty DataFrame
Columns: [count, mean, std, min, 25%, 50%, 75%, max]
Index: []

=== token_count_new ===
Empty DataFrame
Columns: [count, mean, std, min, 25%, 50%, 75%, max]
Index: []

=== token_overlap ===
Empty DataFrame
Columns: [count, mean, 